# Gráficos e cálculos de parâmetros estatísticos e incertezas para dados experimentais

**Autoria:** Ana Luiza de Lima Silva <br> **Disciplina:** Prática em Ciência de Dados - 1° semestre de 2026 <br>
**Instituição:** Ilum escola de ciência

---

## Introdução

A análise de dados obtidos por meio de experimentos constitui uma das principais atividades da prática  laboratorial, sendo essencial para que as informações obtidas possam ser corretamente avaliadas e interpretadas. No entanto, essas medidas constantemente se apresentam associadas à erros e incertezas, resultado de flutuações e perturbações no sistema de medição utilizado. Esses erros podem ser minimizados com a realização de múltiplas medições em uma mesma condição, porém, esse processo demanda uma posterior avaliação e combinação dos dados obtidos.

Assim, análise estatística das medidas, para além de sua coleta, exige organização e interpretação. Visando simplificar o método de tratamento desses dados, o projeto se propõe a realizar o cálculo de parâmetros estatísticos essenciais, como média, variância, desvio padrão, erro padrão, erro relativo e porcentagem de erro, devolvendo essas análises ao usuário e oferecendo a opção de exportá-las. Além disso, também é possível gerar gráficos para a análise dos dados, baseando-se em modelos de ajuste linear, quadrático, exponencial e logarítmico. Esses ajustes baseiam-se em regressões pautadas no Método dos Quadrados Mínimos, sendo possível incorporar a eles barras de erro e bandas de predição e confiança baseadas na distribuição t-Student para dados finitos e variância populacional desconhecida. Associados a eles, são produzidas as equações que descrevem a relação entre os dados em cada um dos modelos, bem como seu coeficiente de determinação (R^2) e caráter da adequação ou não aos dados experimentais fornecidos.

O notebook apresentado a seguir busca propor um modelo que integra o tratamento, interpretação e visualização dos dados experimentais, considerando seus erros e incertezas associados a fim de possibilitar sua avaliação adequada.

---

## Objetivo

Desenvolver uma ferramenta computacional em Python para análise estatística de dados experimentais, gerando tabelas e gráficos visualizáveis e exportáveis com as devidas análises de incertezas associadas. O projeto integra o cálculo parâmetros estatísticos, realização de regressões matemáticas baseadas no Método dos Quadrados Mínimos e geração de barras de erro e de bandas de predição e confiança a fim de automatizar o tratamento e visualização desses dados.

---

## Resultados e discussões

O programa a seguir foi pensado para integrar o recebimento de dados em múltiplos formatos, de acordo com aquele que mais se adeque à demanda do usuário, e o tratamento desses dados e seus parâmetros estatísticos. O usuário recebe gráficos e tabelas conforme as opções que seleciona, podendo exportá-los ou não conforme a sua necessidade.

A organização foi separada em blocos conforme a finalidade de cada um deles, sendo os principais: importação das bibliotecas utilizadas, determinação de funções auxiliares, recebimento dos dados, retorno dos dados e análise de erros no formato de Dataframe e tabelas, e geração dos gráficos.

A seguir, estão todos os códigos utilizados para o projeto, seguidos das respectivas explicações para cada seção.

### Importando bibliotecas

Essa seção realiza a importação de todas as bibliotecas a serem utilizadas durante o projeto. O uso de cada uma delas se deu com as seguintes finalidades:
<ul>
<li><b>pandas:</b> utilizada na etapa de inserção dos dados para leitura dos arquivos em formato .txt ou .xlsx e geração do Dataframe de devolução após o tratamento dos dados;</li>
<br>
<li><b>numpy:</b> utilizada na etapa de geração de gráficos para obter os coeficientes e curvas ajustadas para cada uma das análises;</li>
<br>
<li><b>math:</b> utilizada nas funções auxiliares  e geração dos gráficos para cálculos relacionados à raiz quadrada;</li>
<br>
<li><b>ipywidgets:</b> utilizada na etapa de exportação da tabela e geração dos gráficos para tornar interativa a escolha do tipo de exportação (ou não) a ser feito e o tipo de análise e curva a ser criada;</li>
<br>
<li><b>mathplotlib.pyplot:</b> utilizada para a geração e visualização dos gráficos para operações numéricas e cálculo das regressões;</li>
<br>
<li><b>módulo ast e o objeto literal_eval:</b> objeto utilizado durante a coleta de dados para evitar que as listas de valores informadas sejam interpretadas como uma string, ou seja, converte os dados informados diretamente em uma lista interpretável pelo python;</li>
<br>
<li><b>módulo scipy.stats e o objeto t:</b> utilizado na geração de gráficos para calcular o valor crítico da distribuição dos dados informados, refletindo sua incerteza associada. Utiliza a distribuição t-Student, uma vez que os dados utilizados são finitos e apresentam variância populacional desconhecida, e é aplicada à determinação das bandas de predição e confiança dos gráficos;</li>
<br>
<li><b>módulo IPython.display e o objeto display:</b> utilizado para exibição do Dataframe e das opções de escolha presentes na geração dos gráficos</li>
</ul>

<b>Antes de iniciar o uso do código, garanta que todas essas bibliotecas estão instaladas!!</b>

In [1]:
import pandas as pd
import numpy as np
import math
import ipywidgets as widgets
import matplotlib.pyplot as plt
from ast import literal_eval
from scipy.stats import t
from IPython.display import display, clear_output

### Definindo funções auxiliares

As funções definidas a seguir são utilizadas em múltiplas seções, em especial no tratamento e devolução dos dados estatísticos ao usuário. Seu agrupamento foi realizado para facilitar a leitura e proporcionar fluidez no entendimento do projeto. Apesar de ser possível utilizar bibliotecas para automatizar esses processos, a escolha por definir as funções se deu para inserir no projeto todos os aprendizados efetivados na disciplina.

In [2]:
def media_lista(lista):
    """Calcula o valor médio para os valores presentes em uma lista inserida na lista principal, ou seja, das múltiplas medidas obtidas em uma mesma condição.
    
    Args:
        lista: lista principal de medidas que engloba sublistas com múltiplos valores.
        
    Returns:
        media: lista contendo a média associada a cada lista interna inserida na lista principal.
        """

    media = []

    for elemento in lista:
        media_elemento = sum(elemento)/len(elemento)
        media .append(media_elemento)
    
    return media

In [3]:
def variancia_lista(lista):
    """Calcula a variância populacional ou amostral para cada lista de valores presente na lista principal, ou seja, das múltiplas medidas obtidas em uma mesma condição.

    Args:
        lista: lista principal de medidas que engloba sublistas com múltiplos valores.

    Returns:
        variancias: lista contendo as variâncias associadas a cada lista interna inserida na lista principal.
    """

    variancias = []

    for a in lista:
        n = len(a)
        media = sum(a)/n

        # variável que armazena a soma dos quadrados obtidos para o cálculo da variância
        soma_quadrados_variancia = 0

        for elemento in a:
            soma_quadrados_variancia += (elemento - media)**2

            # define o tipo de variância a ser calculada
            if tipo_variancia == "populacional":
                variancia = soma_quadrados_variancia/n
            elif tipo_variancia == "amostral":
                variancia = soma_quadrados_variancia/(n-1)

        variancias .append(variancia)
        
    return variancias

In [4]:
def desvio_padrao_lista(lista):
    """Calcula o desvio padrão populacional ou amostral para cada lista de valores inserida na lista principal, ou seja, das múltiplas medidas obtidas em uma mesma condição.

    Args:
        lista: lista principal de medidas que engloba sublistas com múltiplos valores.

    Returns:
        desvios: lista contendo os desvios padrões associados a cada lista interna inserida na lista principal.
    """

    desvios = []

    for elemento in lista:
        
        #calculando a média de uma lista interna
        n = len(elemento)
        media = sum(elemento)/n  

        soma_todos_quadrados = []

        for a in elemento:
            soma_quadrados = (a - media)**2

            soma_todos_quadrados .append(soma_quadrados)

            #define o tipo de desvio a ser calculado
            if tipo_desvio == "populacional":
                desvio_elemento = math.sqrt((sum(soma_todos_quadrados))/n)
            elif tipo_desvio == "amostral":
                desvio_elemento = math.sqrt((sum(soma_todos_quadrados))/(n-1))

        desvios .append(desvio_elemento)
        
    return desvios

In [5]:
def erro_padrao_lista(lista):
    """Calcula o erro padrão populacional ou amostral para cada lista inserida na lista principal, ou seja, das múltiplas medidas obtidas em uma mesma condição.
    
    Args:
        lista: lista principal de medidas que engloba sublistas com múltiplos valores.
        
    Returns:
        erros_padroes: lista contendo os erros padrões associados a cada lista interna inserida na lista principal.
        """

    desvios = []

    for elemento in lista:

        # calculando as médias para cada lista interna
        n_elemento = len(elemento)
        media = sum(elemento)/n_elemento  
        soma_todos_quadrados = []

        for a in elemento:

            # calculando o desvio padrão associado a cada lista interna
            soma_quadrados = (a - media)**2
            soma_todos_quadrados .append(soma_quadrados)
            if tipo_desvio == "populacional":
                desvio = math.sqrt((sum(soma_todos_quadrados))/n_elemento)
            elif tipo_desvio == "amostral":
                desvio = math.sqrt((sum(soma_todos_quadrados))/(n_elemento-1))

        desvios .append(desvio)

    erros_padroes = []
    n_lista = len(lista)

    for desv in desvios:
        # calculando os erros padrões associados a cada medida
        erro_pad = desv/math.sqrt(n_lista)
        erros_padroes .append(erro_pad)
    
    return erros_padroes

In [6]:
def erro_relativo_lista(lista):
    """Calcula o erro relativo associado a cada lista presente na lista principal, ou seja, das múltiplas medidas obtidas em uma mesma condição.
    
    Args: 
        lista: lista principal de medidas que engloba sublistas com múltiplos valores.

    Returns:
        erros_relativos: lista contendo os erros relativos associados a cada lista interna inserida na lista principal.
    """

    erros_relativos = []

    for elemento in lista:

        # calculando a média associada a cada lista interna
        media = sum(elemento)/len(elemento)

        # erro absoluto como a diferença entre os extremos de cada lista
        erro_absoluto = max(elemento) - min(elemento)

        # erro relativo como o quociente da divisão entre o erro absoluto e a média
        erro_relativo_elemento = erro_absoluto/media    
        erros_relativos .append(erro_relativo_elemento)

    return erros_relativos

In [7]:
def porcentagem_de_erro_lista(lista):
    """Calcula a porcentagem de erro associada a cada elemento presente em uma lista presente na lista principal, ou seja, das múltiplas medidas obtidas em uma mesma condição.

    Args:
        lista: lista de principal de medidas que engloba sublistas com múltiplos valores.

    Returns:
        porcentagens: lista principal com listas internas contendo a porcentagem de erro associada a cada medida.
    """
    
    porcentagens = []
    for listas_internas in lista:
        media = sum(listas_internas)/len(listas_internas)
        
        erros_internos =[]
        
        for elemento in listas_internas:
            # obtendo as porcentagens de erro para cada lista interna
            porcent_erro = ((elemento - media)/media)*100
            erros_internos .append(porcent_erro)
            
        porcentagens .append(erros_internos)

    return porcentagens

In [8]:
def calcular_r2(y, y_fit):
    """Calcula o R^2 (coeficiente de determinação) associado a um ajuste
    
    Args:
        y: dados experimentais alocados como coordenadas y.
        y_fit: coordenadas y dos pontos previstos pelo ajuste.

    Returns:
        r2: valor obtido para o coeficiente de determinação R^2 do ajuste escolhido.
    """

    # calcula o resíduo entre as medidas experimentais e as do modelo teórico
    SQ_residual = np.sum((y - y_fit)**2)

    # calcula a dispersão dos dados em relação à média 
    SQ_total = np.sum((y - np.mean(y))**2)

    # determina o R^2 como a diferença entre 1 e o quociente da divisão dos resíduos pela dispersão dos dados
    r2 = 1 - SQ_residual/SQ_total
    
    return r2

In [9]:
def avaliar_ajuste(R2, ajuste):
    """Avalia o significado de cada valor R² (coeficiente de determinação), ou seja, o quanto o ajuste realizado concorda com os dados experimentais obtidos.
    
    Args:
        R2: coeficiente de determinação determinado pelo ajuste.
        ajuste: tipo de ajuste.

    Returns:
        mensagens que exibem se a concordância do ajuste com os dados experimentais é excelente, boa, aceitável, baixa ou inadequada.
        """

    if R2 >= 0.98:
        return f"O ajuste {ajuste} apresentou excelente concordância com os dados experimentais."
        
    elif R2 >= 0.95:
        return f"O ajuste {ajuste} apresentou boa concordância com os dados experimentais."

    elif R2 >= 0.90:
        return f"O ajuste {ajuste} apresentou concordância aceitável com os dados experimentais."

    elif R2 >= 0.70:
        return f"Aviso: o ajuste {ajuste} apresentou baixa concordância com os dados experimentais."

    else:
        return f"Aviso: o ajuste {ajuste} não representa adequadamente os dados experimentais."

### Entrada de dados e opções para a análise

A seção a seguir apresenta as principais interações do usuário com o programa, deixando-o responsável pela inserção dos dados que serão considerados ou não para a análise de incertezas e geração dos gráficos. <br><b> Antes de prosseguir, se atente à leitura dos avisos! </b>

##### AVISOS!!!
<ul>
    <li>Atente-se às opções oferecidas, aquelas que não estão explícitas não são aceitas ou reconhecidas. Por exemplo, se para o tipo de arquivo as opções são ".txt", ".xlsx" ou "não", escolha uma delas e garanta a grafia correta;</li>
    <br>
    <li>Caso seu arquivo seja do tipo .txt ou .xlsx, garanta que ele está no mesmo diretório (pasta) que seu notebook. Verifique se o nome inserido está correto e não se esqueça de adicionar após ele o formato, por exemplo: meu_arquivo.txt ou meu_arquivo.xlsx;</li>
    <br>
    <li>Para arquivos em .txt ou .xlsx, verifique e assegure que o nome selecionado para os dados referentes às abscissas e ordenadas coincide com aquele presente no arquivo;</li>
    <br>
    <li>Caso deseje inserir seus dados manualmente, coloque-os no formato de lista, ou seja, entre colchetes (<b>[]</b>);</li>
    <br>
    <li>Medidas inseridas manualmente cujos erros busca-se avaliar devem ser inseridas no formato de listas dentro de listas, ou seja, uma lista principal contendo sublistas com os valores para cada condição avaliada. Exemplo: [[1,2,3], [4,5,6]];</li>
    <br>
    <li>Caso seus dados não apresentem múltiplas medidas para uma mesma condição, ou seja, não está estruturado como listas dentro de uma lista principal, a análise de incertezas não é possível de ser realizada. Ao ser questionado sobre quais dados devem ser considerados para a análise de erros, responda <b>"nenhuma"</b>;</li>
    <br>
    <li>Certifique-se que, caso insira múltiplas medidas para uma mesma condição, a média delas seja diferente de zero. Caso isso não seja contemplado, erros serão produzidos ao tentar calcular as incertezas associadas a elas;</li>
    <br>
    <li>Para quaisquer ajustes que desejar estimar e prever o gráfico, certifique-se de inserir ao menos 4 medidas para que esses possam ser gerados adequadamente. Lembre-se que, quantos mais pontos inserir, mais precisas serão as análises.</li>
</ul>

#### - Quais os tipos de dados?

In [20]:
# reconhece o tipo de dados que o usuário deseja utilizar
dados = input("Seus dados estão em algum arquivo .txt ou .xlsx? Se sim, insira o tipo de arquivo (ex: .txt ou .xlsx), caso contrário, insira 'não'.").lower()

# arquivo no formato .txt
if dados == ".txt":
    
    dados_txt = pd.read_csv(input("Insira o nome do arquivo em .txt:"))
        
    # quais colunas terão seus dados avaliados em cada eixo do gráfico
    coordenadas_x = [literal_eval(v) for v in dados_txt[input("Insira o nome dos dados referentes ao eixo das abscissas (x):")]]
    coordenadas_y = [literal_eval(v) for v in dados_txt[input("Insira o nome dos dados referentes ao eixo das ordenadas (y):")]]

# arquivo no formato .xlsx
elif dados == ".xlsx":
    
    dados_xlsx = pd.read_excel(input("Insira o nome do arquivo em .xlsx:"))
        
    # quais colunas terão seus dados avaliados em cada eixo do gráfico
    coordenadas_x = [literal_eval(v) for v in dados_xlsx[input("Insira o nome dos dados referentes ao eixo das abscissas (x):")]]
    coordenadas_y = [literal_eval(v) for v in dados_xlsx[input("Insira o nome dos dados referentes ao eixo das ordenadas (y):")]]

# o usuário deve adicionar diretamente os dados
elif dados == "não":
    # dados para o eixo das abscissas (x)
    coordenadas_x = literal_eval(input("Digite os valores referentes ao eixo das abscissas (x): "))
    # dados para o eixo das ordenadas (y)
    coordenadas_y = literal_eval(input("Digite os valores de y referentes ao eixo das ordenadas (y): "))

    # erro produzido caso a quantidade de pontos para os eixos sejam diferentes
    if len(coordenadas_x) != len(coordenadas_y):
        raise ValueError("As medidas fornecidas para os eixos contêm quantidades diferentes de pontos. Certifique-se que elas coincidam!")

else:
    raise ValueError("É necessário inserir um tipo de arquivo. Reinicie a célula!")
    

Seus dados estão em algum arquivo .txt ou .xlsx? Se sim, insira o tipo de arquivo (ex: .txt ou .xlsx), caso contrário, insira 'não'. não
Digite os valores referentes ao eixo das abscissas (x):  [1,2,3,4,5]
Digite os valores de y referentes ao eixo das ordenadas (y):  [[0.99,1,1.01],[1.99,2,2.01],[2.99,3,3.01],[3.99,4,4.01],[4.99,5,5.01]]


#### - Quais dados serão considerados para a análise? Quais os tipos de erros avaliados?

In [13]:
# qual(is) dos dados informados serão considerados para a análise de incertezas associadas
informa_dados_analise_erros = input("Indique se os dados dos eixos das abscissas, ordenadas, ambas ou nenhuma devem ser considerados para análise dos erros:").lower()

if informa_dados_analise_erros == "abscissas":
    dados_analise_erros = coordenadas_x
    tipo_variancia = input("Indique se a variância calculada é do tipo populacional ou amostral:").lower()
    tipo_desvio = input("Indique se o desvio a ser calculado é do tipo populacional ou amostral:").lower()
    
elif informa_dados_analise_erros == "ordenadas":
    dados_analise_erros = coordenadas_y
    tipo_variancia = input("Indique se a variância calculada é do tipo populacional ou amostral:").lower()
    tipo_desvio = input("Indique se o desvio a ser calculado é do tipo populacional ou amostral:").lower()
    
elif informa_dados_analise_erros == "ambas":
    dados_analise_erros_x = coordenadas_x
    dados_analise_erros_y = coordenadas_y
    tipo_variancia = input("Indique se a variância calculada é do tipo populacional ou amostral:").lower()
    tipo_desvio = input("Indique se o desvio a ser calculado é do tipo populacional ou amostral:").lower()

elif informa_dados_analise_erros == "nenhuma":
    print("Nenhuma análise de erro será realizada. Não prossiga para a etapa de devolução dos dados. Vá diretamente para a geração do gráfico!")
    
# caso nenhum dos dados seja informado para análise ou houver algum erro, um aviso será gerado
else:
    raise ValueError("O que é isso? A informação não corresponde ao solicitado, se atente às opções disponíveis!")

# gerando erros caso as opções disponíveis não sejam atendidas
if tipo_variancia not in ["populacional", "amostral"]:
        raise ValueError("O tipo de variância selecionado deveria ser populacional ou amostral, reveja sua escolha!")
if tipo_desvio not in ["populacional", "amostral"]:
        raise ValueError("O tipo de desvio selecionado deveria ser populacional ou amostral, reveja sua escolha!")

Indique se os dados dos eixos das abscissas, ordenadas, ambas ou nenhuma devem ser considerados para análise dos erros: ordenadas
Indique se a variância calculada é do tipo populacional ou amostral: populacional
Indique se o desvio a ser calculado é do tipo populacional ou amostral: populacional


#### - Confirmando os dados informados para análise de erros

Caso o dado exibido seja diferente daquele que se deseja avaliar, retorne à etapa anterior e verifique se suas respostas estão compatíveis!

In [14]:
# exibindo os dados considerados para a análise de erros
if informa_dados_analise_erros == "ambas":
    print(f"Os dados considerados para a análise de erros serão: {dados_analise_erros_x} e {dados_analise_erros_y}")
    
elif informa_dados_analise_erros in ["abscissas", "ordenadas"]:
    print(f"Os dados considerados para a análise de erros serão: {dados_analise_erros}")

else:
    print("Nenhum dado será considerado para a análise de erros!")

Os dados considerados para a análise de erros serão: [[0.99, 1, 1.01], [1.99, 2, 2.01], [2.99, 3, 3.01], [3.99, 4, 4.01], [4.99, 5, 5.01]]


#### - Qual a nomenclatura dos elementos do gráfico?

Insira a nomenclatura desejada para os eixos do gráfico e o título. Não se esqueça de adicionar as unidades de medida adequadas a cada dado. 

In [15]:
# nomes para os eixos das ordenadas e abscissas dos gráficos
nome_eixo_abscissas = input("Insira o nome desejado para o eixo das abscissas (x): ")
nome_eixo_ordenadas = input("Insira o nome desejado para o eixo das ordenadas (y): ")

# título desejado para o gráfico
titulo_grafico =  input("Insira o título desejado para o gráfico: ")

Insira o nome desejado para o eixo das abscissas (x):  x
Insira o nome desejado para o eixo das ordenadas (y):  y
Insira o título desejado para o gráfico:  teste


### Devolução dos dados e parâmetros estatísticos

A seguir, caso haja algum dado cuja análise de erros foi solicitada pelo usuário anteriormente, esses serão devolvidos, juntamente com os parâmetros estatísticos a eles associados. O usuário recebe a opção de exportar ou não a tabela produzida, podendo escolher o nome do arquivo caso selecione "sim".

#### - Definindo variáveis

As variáveis a seguir são definidas para facilitar a devolução e posterior acesso das medidas, caso seja necessário.

In [16]:
# variáveis para as análises de erro em apenas um dos eixos
if informa_dados_analise_erros in ["abscissas", "ordenadas"]:
    media = media_lista(dados_analise_erros)
    variancia = variancia_lista(dados_analise_erros)
    desvio_padrao = desvio_padrao_lista(dados_analise_erros)
    erro_padrao = erro_padrao_lista(dados_analise_erros)
    erro_relativo = erro_relativo_lista(dados_analise_erros)
    porcentagem_de_erro = porcentagem_de_erro_lista(dados_analise_erros)

# variáveis para as análises de erro em ambos os eixos
elif informa_dados_analise_erros == "ambas":
    media_x = media_lista(dados_analise_erros_x)
    media_y = media_lista(dados_analise_erros_y)
    variancia_x = variancia_lista(dados_analise_erros_x)
    variancia_y = variancia_lista(dados_analise_erros_y)
    desvio_padrao_x = desvio_padrao_lista(dados_analise_erros_x)
    desvio_padrao_y = desvio_padrao_lista(dados_analise_erros_y)
    erro_padrao_x = erro_padrao_lista(dados_analise_erros_x)
    erro_padrao_y = erro_padrao_lista(dados_analise_erros_y)
    erro_relativo_x = erro_relativo_lista(dados_analise_erros_x)
    erro_relativo_y = erro_relativo_lista(dados_analise_erros_y)
    porcentagem_de_erro_x = porcentagem_de_erro_lista(dados_analise_erros_x)
    porcentagem_de_erro_y = porcentagem_de_erro_lista(dados_analise_erros_y)

#### - Retornando os dados para o usuário

In [17]:
# define o Dataframe de retorno dos dados a ser exibido quando apenas um dos eixos tem suas incertezas avaliadas
if informa_dados_analise_erros in ["abscissas", "ordenadas"]:
    tabela = pd.DataFrame({
        'Média': media,
        'Desvio padrão': desvio_padrao,
        'Variância': variancia,
        'Erro padrão': erro_padrao,
        'Erro relativo': erro_relativo,
        'Porcentagem de erros (%)': porcentagem_de_erro
    })

    # cria uma coluna de Resultados que associa a média dos valores obtidos para uma medida ao seu erro padrão
    tabela['Resultado'] = [f"{m:.3f} ± {e:.3f}" for m,e in zip(media, erro_padrao)]

# define o Dataframe de retorno dos dados a ser exibido quando ambos os eixos tem suas incertezas avaliadas
elif informa_dados_analise_erros == "ambas":
    tabela = pd.DataFrame({
        'Média dos dados das abscissas': media_x,
        'Média dos dados das ordenadas': media_y,
        'Desvio padrão dos dados das abscissas': desvio_padrao_x,
        'Desvio padrão dos dados das ordenadas': desvio_padrao_y,
        'Variância dos dados das abscissas': variancia_x,
        'Variância dos dados das ordenadas': variancia_y,
        'Erro padrão dos dados das abscissas': erro_padrao_x,
        'Erro padrão dos dados das ordenadas': erro_padrao_y,
        'Erro relativo dos dados das abscissas': erro_relativo_x,
        'Erro relativo dos dados das ordenadas': erro_relativo_y,
        'Porcentagem de erros dos dados das abscissas (%)': porcentagem_de_erro_x,
        'Porcentagem de erros dos dados das ordenadas (%)': porcentagem_de_erro_y,
    })

    # criação colunas de Resultados que associam a média obtida para cada medida ao seu erro padrão
    tabela['Resultado abscissas'] = [f"{m:.3f} ± {e:.3f}" for m, e in zip(media_x, erro_padrao_x)]
    tabela['Resultado ordenadas'] = [f"{m:.3f} ± {e:.3f}" for m, e in zip(media_y, erro_padrao_y)]

# define o Dataframe de devolução dos dados caso nenhuma incerteza seja avliada
elif informa_dados_analise_erros == "nenhuma":
    tabela = pd.DataFrame({"Dados para o eixo das abscissas": coordenadas_x,
                          "Dados para o eixo das ordenadas": coordenadas_y})

# estabelecendo o índice exibido para cada medida
tabela.index = [f"Medida {i}" for i in range(1, len(tabela)+1)]

# determinando o layout da tabela
display(tabela
    .style.set_caption("Análise estatística das medições") # determinando o titulo da tabela
    .set_properties(**{'text-align': 'center'})  # alinhando as informações no centro de cada coluna
    .format(precision=4) # arredondando as casas decimais de todos as medidas apresentadas
       )

,Média,Desvio padrão,Variância,Erro padrão,Erro relativo,Porcentagem de erros (%),Resultado
Medida 1,1.0000,0.0082,0.0001,0.0037,0.0200,"[-1.0000000000000009, 0.0, 1.0000000000000009]",1.000 ± 0.004
Medida 2,2.0000,0.0082,0.0001,0.0037,0.0100,"[-0.5000000000000004, 0.0, 0.49999999999998934]",2.000 ± 0.004
Medida 3,3.0000,0.0082,0.0001,0.0037,0.0067,"[-0.33333333333332626, 0.0, 0.33333333333332626]",3.000 ± 0.004
Medida 4,4.0000,0.0082,0.0001,0.0037,0.0050,"[-0.24999999999999467, 0.0, 0.24999999999999467]",4.000 ± 0.004
Medida 5,5.0000,0.0082,0.0001,0.0037,0.0040,"[-0.19999999999999576, 0.0, 0.19999999999999576]",5.000 ± 0.004


<br>Para a seleção das opções de exportação da tabela e escolha do nome associado ao arquivo, foi utilizada a biblioteca ipywidgets para criar botões e opções interativas para seleção das opções, visando melhorar a experiência do usuário. O mesmo se deu durante a geração e devolução dos gráficos.

In [18]:
"""Definindo as opções de escolha do ususário ao salvar a tabela (ou não)."""

# botão de escolha para a exportação
exportar_tabela = widgets.ToggleButtons(
    options=['não','sim'],
    description='Exportar a tabela para o formato .xlsx?'
)

# campo de escolha do nome do arquivo (inicialmente oculto)
nome_arquivo_tabela = widgets.Text(
    value='tabela_estatistica.xlsx',
    description='Nome do arquivo:'
)
nome_arquivo_tabela.layout.display = 'none'

# botão de confirmação
botao_exportar = widgets.Button(
    description='Confirmar',
    button_style='success',
    icon='save'
)

# saída dos dados
saida_exportacao = widgets.Output()

def alterar_visibilidade(alteracao):
    """Define se o campo de inserção do nome do arquivo para exportação da tabela será exibido ou ocultado."""
    
    if alteracao['new'] == 'sim':
        nome_arquivo_tabela.layout.display = 'block'
    else:
        nome_arquivo_tabela.layout.display = 'none'

# observando mudanças na opção de exportação para definir a exibição ou não do campo de inserção do nome
exportar_tabela.observe(alterar_visibilidade, names='value')

# função de exportação da tabela
def confirmar_exportacao(b):
    """Exibe a mensagem correspondente à opção de exportação selecionada e salva o Dataframe no formato .xlsx.
    
    Args:
        opção selecionada no botão "exportar_tabela", ou seja, "sim" ou "não".

    Returns:
        caso a entrada seja "sim", o gráfico será exportado e a mensagem de exportação com o nome selecionado será exibida.
        caso a entrada seja "não", o gráfico não será exportado e uma mensgaem condizente será exibida.
    """
    
    # estabelecendo a interface de saída para exibir as mensagens
    with saida_exportacao:

        clear_output()

        # obtendo nome desejado para o arquivo
        if exportar_tabela.value == 'sim':
            nome = nome_arquivo_tabela.value

            # adicionando o formato do arquivo ao seu nome
            if not nome.endswith('.xlsx'):
                nome += '.xlsx'

            # tentativa de exportar a tabela, caso a selação seja "sim"
            try:
                tabela.to_excel(nome, index=True)
                print(f"Tabela exportada como '{nome}'")

            except Exception as erro:
                print(f"Erro ao exportar: {erro}")

        # caso em que a seleção é "não"
        else:
            print("Tabela não exportada.")

# associando botão à confirmação
botao_exportar.on_click(confirmar_exportacao)

# interface
interface_exportacao = widgets.VBox([
    exportar_tabela,
    nome_arquivo_tabela,
    botao_exportar,
    saida_exportacao
])

#exibindo a interface
display(interface_exportacao)

### Geração dos gráficos

Para a visualização do gráfico e seleção das opções para o tipo de ajuste a ser realizado e banda de avaliação a ser gerada, foi utilizada a biblioteca ipywidgets para tornar esse processo interativo, melhorar e guiar a utilização pelo usuário.

Com base no tipo de ajuste selecionado, é calculada e devolvida a equação a ele associada, bem como seu coeficiente de determinação (R²) e adequação do ajuste aos dados experimentais fornecidos. Além disso, o usuário pode selecionar a geração de bandas de confiança, bandas mais estreitas que exibem a confiabilidade do parâmetro escolhido paraa determinado ponto, ou bandas de predição, bandas mais largas que buscam prever as posições de dados experimentais futuros. Para esses cálculos, foi utilizada a distribuição t-student, distribuição de probabilidade contínua e simétrica que se adequa a dados finitos para os quais a variância populacional é desconhecida.

Conforme a opção selecionada para a análise de incertezas, barras de erro correspondentes ao desvio padrão associado às medidas são adicionadas aos pontos experimentais, fornecendo uma etapa adicional para interpretação da confibilidade dos dados.

Para a determinação de todos os ajustes, baseou-se no método dos quadrados mínimos, implementado-o a partir das funções de ajuste já presentes na biblioteca numpy. Assim, por meio do objeto polyfit(), procurou-se obter os coeficientes para o ajuste que minimizem o quadrado dos resíduos, ou seja, da diferença entre os dados experimentais apresentados e os dados teóricos propostos pelo ajuste, expressos no código por meio da variável s. Além disso, esse objeto também foi utilizado para obter a matriz de covariância associada ao ajuste, a qual permite estimar as incertezas associadas aos parâmetros do ajuste.

Para o ajuste linear e quadrático, a regressão foi diretamente aplicada aos dados fornecidos, uma vez que esses já são lineares. Enquanto isso, para o ajuste exponencial e logarítmico, foi realizada a linearização das equações a fim de utilizar o mesmo método de análise baseado em regressões lineares. Para o ajuste exponencial, isso foi realizado aplicando o logarítmo natural às medidas em y, de modo que ln(y) = ln(a) + bx, sendo posteriormente necessário desfazer essa transformação para devolver os dados da equação corretamente ao usuário, ou seja, ae^(bx). Já para o ajuste logarítmico, a aplicação do logarítmo natural se deu diretamente aos valores de x, permitindo a implementação aos dados finais impressos para o usuário.

A seguir, os gráficos são produzidos conforme as especificações do usuário. É possível transicionar entre opções até que aquela que mais se adeque às demandas seja selecionada, podendo qualquer uma delas ser exportada como uma imagem se o usuário assim desejar.

In [19]:
"""Define as opções de escolha que o usuário pode fazer quanto aos gráficos produzidos."""

tipo_ajuste = widgets.Dropdown(
    options=[
        'linear',
        'quadrático',
        'exponencial',
        'logarítmico'
    ],
    value='linear',
    description='Ajuste:'
)

tipo_banda = widgets.Dropdown(
    options=[
        'nenhuma',
        'confiança',
        'predição',
        'ambas'
    ],
    value='ambas',
    description='Bandas:'
)

botao_salvar = widgets.ToggleButtons(
    options=['não', 'sim'],
    description='Salvar a imagem no formato .png?',
    button_style='',
)

nome_arquivo_grafico = widgets.Text(
    value='grafico.png',
    description='Nome da imagem:'
)
nome_arquivo_grafico.layout.display = 'none'

botao_gerar = widgets.Button(
    description='Gerar gráfico',
    button_style='success'
)

# estabelece a interface de saída para exibir as interações
saida = widgets.Output()

def mostrar_campo(alteracao):
    """Função que determina a exibição ou não do campo de inserção para o nome da imagem a ser exportada.
    
    Args:
        alteracao: opção selecionada no botão "botao_salvar".

    Returns:
        caso a opção seja "sim", será exibida a caixa de inserção do nome do imagem.
        caso a opção seja "não", a caixa de inserção continuará oculta.
        """
    
    if alteracao['new'] == 'sim':
        nome_arquivo_grafico.layout.display = 'block'
    else:
        nome_arquivo_grafico.layout.display = 'none'

botao_salvar.observe(mostrar_campo, names='value')


def gerar_grafico(b):
    """Função que define os parâmetros para geração dos gráficos.

    Args:
        b: opção de ajuste selecionada no botão "tipo_ajuste"

    Returns:
        gráfico e parâmetros correspondentes ao ajuste selecionado
    """

    # associa o gráfico produzido à interface de saída
    with saida:

        # a imagem anterior só é substituída quando a nova imagem é gerada
        clear_output(wait=True)

        # valor para o tipo de ajuste e banda escolhido pelo usuário
        ajuste = tipo_ajuste.value
        banda = tipo_banda.value

        # definindo tabela de cores acessível para daltônicos a ser utilizada nos gráficos
        cores_okabe_ito = [
            '#56B4E9', # banda de predição
            '#0072B2', # banda de confiança
            '#000000', # pontos e barras de erro
            '#D55E00' # ajuste
        ]
        
        """Definindo o gráfico para cada escolha do usuário"""
        if ajuste == 'linear':

            """Definindo os gráficos com base em cada possibilidade de eixos avaliados para a análise de erros"""
            if informa_dados_analise_erros == "abscissas":

                # obtendo os coeficientes e matriz de covariância
                coeficientes, covariancia = np.polyfit(media, coordenadas_y, 1, cov=True)
                
                # gerando pontos adicionais para suavizar a curva
                x_suave = np.linspace(min(media), max(media), 500)
                y_suave = np.polyval(coeficientes, x_suave)

                # curva ajustada
                y_fit = np.polyval(coeficientes, media)

                # parâmetros utilizados para obter os limites das bandas de confiança e predição
                n = len(media)
                x_medio = np.mean(media)
                residuos = coordenadas_y - y_fit
                x_medio = np.mean(media)

                # dispersão dos valores de x
                Sxx = np.sum((media - x_medio)**2)

                # coeficiente de determinação
                R2 = calcular_r2(coordenadas_y, y_fit)
                print(f"R² = {R2:.6f}")
                
                # resoluções e dimensões da imagem
                plt.figure(figsize=(9,6), dpi=120)

                # plotando a curva para o ajuste linear
                plt.plot(x_suave, y_suave, label='Ajuste linear', color = '#D55E00')

                # plotando os pontos experimentais
                plt.scatter(media, coordenadas_y, label='Dados experimentais', zorder=3, color='#000000')

                # barras de erros para as coordenadas em x com base no desvio padrão
                plt.errorbar(media, coordenadas_y, xerr=desvio_padrao, fmt='none', elinewidth=1.5, alpha=0.9, capsize=5, label='Desvio padrão', ecolor='#000000')

            
            elif informa_dados_analise_erros == "ordenadas":

                # obtendo os coeficientes e matriz de covariância
                coeficientes,covariancia = np.polyfit(coordenadas_x, media, 1, cov=True)

                # gerando pontos adicionais para suavizar a curva
                x_suave = np.linspace(min(coordenadas_x), max(coordenadas_x), 500)
                y_suave = np.polyval(coeficientes, x_suave)
                
                # curva ajustada
                y_fit = np.polyval(coeficientes, coordenadas_x)

                # parâmetros utilizados para obter os limites das bandas de confiança e predição
                n = len(coordenadas_x)
                x_medio = np.mean(coordenadas_x)
                residuos = media - y_fit
                x_medio = np.mean(coordenadas_x)

                # dispersão dos valores de x
                Sxx = np.sum((coordenadas_x - x_medio)**2)

                # coeficiente de determinação
                R2 = calcular_r2(media, y_fit)
                print(f"R² = {R2:.6f}")
                
                # dimensões e resolução da figura
                plt.figure(figsize=(9,6), dpi=120)

                # plotando a curva ajustada
                plt.plot(x_suave, y_suave, label='Ajuste linear', color = '#D55E00')

                # plotando os dados experimentais
                plt.scatter(coordenadas_x, media, label='Dados experimentais', zorder=3, color='#000000')

                # barras de erro para as coordenadas do eixo y com base no desvio padrão
                plt.errorbar(coordenadas_x, media, yerr=desvio_padrao, fmt='none', elinewidth=1.5, alpha=0.9, capsize=5, label='Desvio padrão', ecolor='#000000')
                
            elif informa_dados_analise_erros == "ambas":

                # obtendo os coeficientes e matriz de covariância
                coeficientes, covariancia = np.polyfit(media_x, media_y, 1, cov=True)

                # gerando pontos adicionais para suavizar a curva
                x_suave = np.linspace(min(media_x), max(media_x), 500)
                y_suave = np.polyval(coeficientes, x_suave)
                
                # curva ajustada
                y_fit = np.polyval(coeficientes, media_x)

                # parâmetros utilizados para obter os limites das bandas de confiança e predição
                n = len(media_x)
                x_medio = np.mean(media_x)
                residuos = media_y - y_fit
                x_medio = np.mean(media_x)

                # dispersão das medidas de x
                Sxx = np.sum((media_x - x_medio)**2)

                # coeficiente de determinação
                R2 = calcular_r2(media_y, y_fit)
                print(f"R² = {R2:.6f}")

                # dimensões e reslução da imagem
                plt.figure(figsize=(9,6), dpi=120)
                
                # plotando a cruva ajustada
                plt.plot(x_suave, y_suave, label='Ajuste linear', color = '#D55E00')

                # plotando os pontos experimentais
                plt.scatter(media_x, media_y, label='Dados experimentais', zorder=3, color='#000000')

                # barras de erro associadas a ambos os eixos
                plt.errorbar(media_x, media_y, xerr=desvio_padrao_x, yerr=desvio_padrao_y, fmt='none', elinewidth=1.5, alpha=0.9, capsize=5, label='Desvio padrão', ecolor='#000000')

            elif informa_dados_analise_erros == "nenhuma":

                # obtendo os coeficientes e matriz de covariância
                coeficientes, covariancia = np.polyfit(coordenadas_x, coordenadas_y, 1, cov=True)

                # gerando pontos adicionais para suavizar a curva
                x_suave = np.linspace(min(coordenadas_x), max(coordenadas_x), 500)
                y_suave = np.polyval(coeficientes, x_suave)
                
                # curva ajustada
                y_fit = np.polyval(coeficientes, coordenadas_x)

                # coeficiente de determinação
                R2 = calcular_r2(coordenadas_y, y_fit)
                print(f"R² = {R2:.6f}")
                
                 # dimensões e reslução da imagem
                plt.figure(figsize=(9,6), dpi=120)
                
                # plotando a cruva ajustada
                plt.plot(x_suave, y_suave, label='Ajuste linear', color = '#D55E00')

                # plotando os pontos experimentais
                plt.scatter(coordenadas_x, coordenadas_y, label='Dados experimentais', zorder=3, color='#000000')

            # incertezas para os parâmetros a e b com base na matriz de covariância
            erro_a = np.sqrt(covariancia[0,0])
            erro_b = np.sqrt(covariancia[1,1])
            
            a, b = coeficientes
            print(f"Os coeficientes obtidos para a equação de reta ax + b são: a = {a:.3f} ± {erro_a:.3f}, b = {b:.3f} ± {erro_b:.3f}")

            if informa_dados_analise_erros in ["ordenadas", "abscissas", "ambas"]:
                """Determinando os limites superiores e inferiores para as bandas de predição e confiança."""
                
                graus_liberdade = n-2
                confianca = 0.95
                
                # dispersão dos resíduos para a curva ajustada
                s = np.sqrt(np.sum(residuos**2)/(graus_liberdade))

                # usa distribuição t de Student para obter o valor crítico da distribuição dos dados
                t_critico = t.ppf((1+confianca)/2, graus_liberdade)

                # erro associado à predição
                erro_pred = s*np.sqrt(1+1/n +(x_suave - x_medio)**2/Sxx)

                pred_superior = y_suave + t_critico*erro_pred
                pred_inferior = y_suave - t_critico*erro_pred

                # erro associado à banda de confiança
                erro_ic = s*np.sqrt(1/n + (x_suave - x_medio)**2/Sxx)

                ic_superior = y_suave + t_critico*erro_ic
                ic_inferior = y_suave - t_critico*erro_ic
                
            print(f'Equação: f(x) = ({a:.3f} ± {erro_a:.3f})x + ({b:.3f} ± {erro_b:.3f})')
            
            ajuste = "linear"
            print(f'{avaliar_ajuste(R2,ajuste)}')

        
        elif ajuste == 'quadrático':

            """Definindo os gráficos com base em cada possibilidade de eixos avaliados para a análise de erros"""
            if informa_dados_analise_erros == "abscissas":

                # obtendo os coeficientes e matriz de covariância
                coeficientes, covariancia = np.polyfit(media, coordenadas_y, 2, cov = True)

                # gerando mais pontos no eixo x e y para suavizar a curva
                x_suave = np.linspace(min(media), max(media), 500)
                y_suave = np.polyval(coeficientes, x_suave)
                y_fit = np.polyval(coeficientes, media)

                # parâmetros utilizados para obter os limites das bandas de confiança e predição
                residuos = coordenadas_y - y_fit
                n = len(media)
                x_medio = np.mean(media)

                # dispersão das medidas do eixo x
                Sxx = np.sum((media - x_medio)**2)

                # coeficiente de determinação
                R2 = calcular_r2(coordenadas_y, y_fit)
                print(f"R² = {R2:.6f}")

                # dimensão e resolução da imagem
                plt.figure(figsize=(9,6), dpi=120)

                # plotando a curva ajustada
                plt.plot(x_suave, y_suave, label='Ajuste quadrático', color = '#D55E00')

                # plotando os pontos experimentais
                plt.scatter(media, coordenadas_y, label='Dados experimentais', zorder=3, color='#000000')

                # barras de erro para as coordenadas em x com base no desvio padrão
                plt.errorbar(media, coordenadas_y, xerr=desvio_padrao, fmt='none', elinewidth=1.5, alpha=0.7, capsize=5, label='Desvio padrão', ecolor='#000000')
                
            elif informa_dados_analise_erros == "ordenadas":

                # obtendo os coeficientes e matriz de covariância
                coeficientes, covariancia = np.polyfit(coordenadas_x, media, 2, cov=True)

                # gerando mais pontos no eixo x e y para suavizar a curva
                x_suave = np.linspace(min(coordenadas_x), max(coordenadas_x), 500)
                y_suave = np.polyval(coeficientes, x_suave)

                # curva ajustada
                y_fit = np.polyval(coeficientes, coordenadas_x)

                # parâmetros utilizados para obter os limites das bandas de confiança e predição
                residuos = media - y_fit
                n = len(coordenadas_x)
                x_medio = np.mean(coordenadas_x)

                # dispersão das medidas no eixo x
                Sxx = np.sum((coordenadas_x - x_medio)**2)

                # coeficiente de determinação
                R2 = calcular_r2(media, y_fit)
                print(f"R² = {R2:.6f}")
            
                # dimensões e reolução da imagem
                plt.figure(figsize=(9,6), dpi=120)

                # plotando a curva ajustada
                plt.plot(x_suave, y_suave, label='Ajuste quadrático', color = '#D55E00')

                # plotando os dados experimentais
                plt.scatter(coordenadas_x, media, label='Dados experimentais', zorder=3, color='#000000')

                # barras de erro para as coordenadas em y com base no desvio padrão
                plt.errorbar(coordenadas_x, media, yerr=desvio_padrao, fmt='none', elinewidth=1.5, alpha=0.7, capsize=5, label='Desvio padrão', ecolor='#000000')
                
            elif informa_dados_analise_erros == "ambas":

                # obtendo os coeficientes e matriz de covariância
                coeficientes, covariancia = np.polyfit(media_x, media_y, 2, cov=True)

                # gerando mais pontos no eixo x e y para suavizar a curva
                x_suave = np.linspace(min(media_x), max(media_x), 500)
                y_suave = np.polyval(coeficientes, x_suave)

                # curva ajustada
                y_fit = np.polyval(coeficientes, media_x)

                # parâmetros utilizados para obter os limites das bandas de confiança e predição
                residuos = media_y - y_fit
                n = len(media_x)
                x_medio = np.mean(media_x)

                # dispersão das medidas no eixo x
                Sxx = np.sum((media_x - x_medio)**2)

                # coeficiente de determinação
                R2 = calcular_r2(media_y, y_fit)
                print(f"R² = {R2:.6f}")

                # dimensões e resolução da imagem
                plt.figure(figsize=(9,6), dpi=120)

                # plotando a curva ajustada
                plt.plot(x_suave, y_suave, label='Ajuste quadrático', color = '#D55E00')

                # plotando os dados experimentais
                plt.scatter(media_x, media_y, label='Dados experimentais', zorder=3, color='#000000')

                # barras de erro para ambos os eixos com base no desvio padrão
                plt.errorbar(media_x, media_y, xerr=desvio_padrao_x, yerr=desvio_padrao_y, fmt='none', elinewidth=1.5, alpha=0.7, capsize=5, label='Desvio padrão', ecolor='#000000')

            elif informa_dados_analise_erros == "nenhuma":

                # obtendo os coeficientes e matriz de covariância
                coeficientes, covariancia = np.polyfit(coordenadas_x, coordenadas_y, 2, cov=True)

                # gerando mais pontos no eixo x e y para suavizar a curva
                x_suave = np.linspace(min(coordenadas_x), max(coordenadas_x), 500)
                y_suave = np.polyval(coeficientes, x_suave)
                y_fit = np.polyval(coeficientes, coordenadas_x)

                # coeficiente de determinação
                R2 = calcular_r2(coordenadas_y, y_fit)
                print(f"R² = {R2:.6f}")

                # dimensões e reolução da imagem
                plt.figure(figsize=(9,6), dpi=120)

                # plotando a curva ajustada
                plt.plot(x_suave, y_suave, label='Ajuste quadrático', color = '#D55E00')

                # plotando os dados experimentais
                plt.scatter(coordenadas_x, coordenadas_y, label='Dados experimentais', zorder=3, color='#000000')

            # obtendo as incertezas associados aos parâmetros a, b e c com base na matriz de covariância
            erro_a = np.sqrt(covariancia[0,0])
            erro_b = np.sqrt(covariancia[1,1])
            erro_c = np.sqrt(covariancia[2,2])
            
            a, b, c = coeficientes
            print(f"Os coeficientes obtidos para a equação da parábola ax² + bx + c são: a= {a:.3f} ± {erro_a:.3f}, b = {b:.3f} ± {erro_b:.3f}, c = {c: .3f} ± {erro_c:.3f}")
            
            print(f'Equação = y = ({a:.3f} ± {erro_a:.3f})x² + ({b:.3f} ± {erro_b:.3f})x + ({c:.3f} ± {erro_c:.3f})')
            
            ajuste = "quadrático"
            print(f'{avaliar_ajuste(R2,ajuste)}')

            if informa_dados_analise_erros in ["abscissas", "ordenadas", "ambas"]:
                """Determinando os limites superiores e inferiores para as bandas de predição e confiança."""
                # dispersão dos resíduos apra a curva ajustada
                graus_liberdade = n-3
                s = np.sqrt(np.sum(residuos**2)/(graus_liberdade))
                confianca = 0.95
    
                # uso da distribuição t de Student para obter o valor crítico da distribuição
                t_critico = t.ppf((1+confianca)/2, graus_liberdade)
    
                # erro associado à predição
                erro_pred = s*np.sqrt(1+1/n +(x_suave - x_medio)**2/Sxx)
                pred_superior = y_suave + t_critico*erro_pred
                pred_inferior = y_suave - t_critico*erro_pred
    
                # erro associado à confiança
                erro_ic = s*np.sqrt(1/n + (x_suave - x_medio)**2/Sxx)
                ic_superior = y_suave + t_critico*erro_ic
                ic_inferior = y_suave - t_critico*erro_ic
                

        elif ajuste == 'exponencial':
            if informa_dados_analise_erros == "abscissas":

                # mensagem de aviso caso algum dado considerado para o exponencial seja negativo
                if any(y <= 0 for y in coordenadas_y):
                    raise ValueError("Não é possível realizar ajuste exponencial com y <= 0")

                # aplicando o logarítmo natural aos valores de y para linearizar a regressão
                ln_y = np.log(coordenadas_y)

                # obtendo os coeficientes e matriz de covariância
                coeficientes, covariancia = np.polyfit(media, ln_y, 1, cov=True)
                b, ln_a = coeficientes

                # aplicando o exponencial ao ln(a) para obtê-lo novamente
                a = np.exp(ln_a)

                # curva exponencial com base nos dados experimentais
                y_fit = a*np.exp(b*np.array(media))

                # curva ajustada
                ln_y_fit = np.polyval(coeficientes, media)

                # gerando pontos adicionais para suavizar a curva
                x_suave = np.linspace(min(media), max(media), 500)
                y_suave = a*np.exp(b*x_suave)

                # parâmetros utilizados para obter os limites das bandas de confiança e predição
                n = len(media)
                x_medio = np.mean(media)

                # dispersão das medidas no eixo x
                Sxx = np.sum((media - x_medio)**2)

                # coeficiente de determinação
                R2 = calcular_r2(coordenadas_y, y_fit)
                print(f"R² = {R2:.6f}")
                
                # dimensão e resolução da imagem
                plt.figure(figsize=(9,6), dpi=120)

                # plotando a curva ajustada
                plt.plot( x_suave, y_suave, label='Ajuste exponencial', color = '#D55E00')

                # plotando os dados experimentais
                plt.scatter(media, coordenadas_y, label='Dados experimentais', zorder=3, color='#000000')

                # barras de erro associados às medidas no eixo x
                plt.errorbar(media, coordenadas_y, xerr=desvio_padrao, fmt='none', elinewidth=1.5, alpha=0.7, capsize=5, label='Desvio padrão', ecolor='#000000')
                
            elif informa_dados_analise_erros == "ordenadas":
                # mensagem de aviso caso algum dado considerado para o exponencial seja negativo
                if any(y <= 0 for y in media):
                    raise ValueError("Não é possível realizar ajuste exponencial com y <= 0")

                # aplicando o logarítmo natural aos valores de y para linearizar a regressão
                ln_y = np.log(media)

                # obtendo os coeficientes e matriz de covariância
                coeficientes, covariancia = np.polyfit(coordenadas_x, ln_y, 1, cov=True)
                b, ln_a = coeficientes

                # aplicando o exponencial ao ln(a) para obtê-lo novamente
                a = np.exp(ln_a)

                # curva exponencial com base nos dados experimentais
                y_fit = a*np.exp(b*np.array(coordenadas_x))

                # curva ajustada
                ln_y_fit = np.polyval(coeficientes, coordenadas_x)

                # gerando pontos adicionais para suavizar a curva
                x_suave = np.linspace(min(coordenadas_x), max(coordenadas_x), 500)
                y_suave = a*np.exp(b*x_suave)

                # parâmetros utilizados para obter os limites das bandas de confiança e predição
                n = len(coordenadas_x)
                x_medio = np.mean(coordenadas_x)

                # dispersão das medidas no eixo x
                Sxx = np.sum((coordenadas_x - x_medio)**2)

                # coeficiente de determinação
                R2 = calcular_r2(media, y_fit)
                print(f"R² = {R2:.6f}")
                
                # dimensões e resolução da imagem
                plt.figure(figsize=(9,6), dpi=120)

                #plotando curva ajustada
                plt.plot( x_suave, y_suave, label='Ajuste exponencial', color = '#D55E00')

                # plotando dados experimentais
                plt.scatter(coordenadas_x, media, label='Dados experimentais', zorder=3, color='#000000')

                # barras de erro para os dados do eixo x com base no desvio padrão
                plt.errorbar(coordenadas_x, media, yerr=desvio_padrao, fmt='none', elinewidth=1.5, alpha=0.7, capsize=5, label='Desvio padrão', ecolor='#000000')
                
            elif informa_dados_analise_erros == "ambas":
                # mensagem de aviso caso algum dado considerado para o exponencial seja negativo
                if any(y <= 0 for y in media_y):
                    raise ValueError("Não é possível realizar ajuste exponencial com y <= 0")

                # aplicando o logarítmo natural às medidas de y para linearizar a regressão
                ln_y = np.log(media_y)

                # obtendo os coeficientes e matriz de covariância
                coeficientes, covariancia = np.polyfit(media_x, ln_y, 1, cov=True)
                b, ln_a = coeficientes

                # aplicando o exponencial a ln(a) para obtê-lo novamente
                a = np.exp(ln_a)

                # curva exponencial com base nos dados experimentais
                y_fit = a*np.exp(b*np.array(media_x))

                # curva ajustada
                ln_y_fit = np.polyval(coeficientes, media_x)

                # criando pontos adicionais para suavizar a curva
                x_suave = np.linspace(min(media_x), max(media_x), 500)
                y_suave = a*np.exp(b*x_suave)

                # parâmetros utilizados para obter os limites das bandas de confiança e predição
                n = len(media_x)
                x_medio = np.mean(media_x)

                # dispersão associada às medidas no eixo x
                Sxx = np.sum((media_x - x_medio)**2)

                # coeficiente de determinação
                R2 = calcular_r2(media_y, y_fit)
                print(f"R² = {R2:.6f}")
                
                # dimensões e resolução da imagem
                plt.figure(figsize=(9,6), dpi=120)

                # plotando a curva ajustada
                plt.plot( x_suave, y_suave, label='Ajuste exponencial', color = '#D55E00')

                # plotando os dados experimentais
                plt.scatter(media_x, media_y, label='Dados experimentais', zorder=3, color='#000000')

                # barras de erro associadas a ambos os eixos com base no desvio padrão
                plt.errorbar(media_x, media_y, xerr=desvio_padrao_x, yerr=desvio_padrao_y, fmt='none', elinewidth=1.5, alpha=0.7, capsize=5, label='Desvio padrão', ecolor='#000000')

            if informa_dados_analise_erros == "nenhuma":
                # mensagem de aviso caso algum dado considerado para o exponencial seja negativo
                if any(y <= 0 for y in coordenadas_y):
                    raise ValueError("Não é possível realizar ajuste exponencial com y <= 0")

                # aplicando o logarítmo natural às medidas de y para linearizar a regressão
                ln_y = np.log(coordenadas_y)

                # obtendo os coeficientes e matriz de covariância
                coeficientes, covariancia = np.polyfit(coordenadas_x, ln_y, 1, cov=True)
                b, ln_a = coeficientes

                # aplicando o exponencial a ln(a) para obtê-lo novamente
                a = np.exp(ln_a)

                # curva exponencial com base nos dados experimentais
                y_fit = a*np.exp(b*np.array(coordenadas_x))

                # curva ajustada
                ln_y_fit = np.polyval(coeficientes, coordenadas_x)

                # criando pontos adicionais para suavizar a curva
                x_suave = np.linspace(min(coordenadas_x), max(coordenadas_x), 500)
                y_suave = a*np.exp(b*x_suave)

                # coeficiente de determinação
                R2 = calcular_r2(coordenadas_y, y_fit)
                print(f"R² = {R2:.6f}")

                # dimensões e resolução da imagem
                plt.figure(figsize=(9,6), dpi=120)

                # plotando a curva ajustada
                plt.plot(x_suave, y_suave, label='Ajuste exponencial', color = '#D55E00')

                # plotando os dados experimentais
                plt.scatter(coordenadas_x, coordenadas_y, label='Dados experimentais', zorder=3, color='#000000')
                

            # obtendo as incertezas associadas aos parâmetros a e b com base na matriz de covariância
            erro_b = np.sqrt(covariancia[0,0])
            erro_ln_a = np.sqrt(covariancia[1,1])
            erro_a = a*erro_ln_a
            
            print(f"Os coeficientes obtidos para a equação ae^(bx) que expressa essa função exponencial são: a = {a:.3f} ± {erro_a:.3f}, b = {b:.3f} ± {erro_b:.3f}")
            
            print(f'Equação: y = ({a:.3f} ± {erro_a:.3f})e^[({b:.3f} ± {erro_b:.3f})x]')
            
            ajuste = "exponencial"
            print(f'{avaliar_ajuste(R2,ajuste)}')

            if informa_dados_analise_erros in ["abscissas", "ordenadas", "ambas"]:
                # curva ajustada com bases em múltiplos pontos para x
                ln_y_suave = np.polyval(coeficientes, x_suave)
                residuos = ln_y - ln_y_fit
                
                graus_liberdade = n-2
                confianca = 0.95
    
                # dispersão dos resíduos para a curva ajustada
                s = np.sqrt(np.sum(residuos**2)/(graus_liberdade))
    
                # uso da distribuição t de Student para obter o valor crítico da distribuição de dados
                t_critico = t.ppf((1+confianca)/2, graus_liberdade)
    
                # erro associado à predição
                erro_pred = s*np.sqrt(1+1/n +(x_suave - x_medio)**2/Sxx)
    
                # limites da predição no espaço linearizado
                ln_pred_superior = ln_y_suave + t_critico*erro_pred
                ln_pred_inferior = ln_y_suave - t_critico*erro_pred
    
                # limites da predição no espaço exponencial
                pred_superior = np.exp(ln_pred_superior)
                pred_inferior = np.exp(ln_pred_inferior)
    
                # erro associado à confiança
                erro_ic = s*np.sqrt(1/n + (x_suave - x_medio)**2/Sxx)
    
                # limites de confiança no espaço linearizado
                ln_ic_superior = ln_y_suave + t_critico*erro_ic
                ln_ic_inferior = ln_y_suave - t_critico*erro_ic
    
                # limites de confiança no espaço exponencial
                ic_superior = np.exp(ln_ic_superior)
                ic_inferior = np.exp(ln_ic_inferior)
            

        elif ajuste == 'logarítmico':
            # mensagem de aviso caso algum dado considerado para o logarítmo seja negativo
            if informa_dados_analise_erros == "abscissas":
                if any(x <= 0 for x in media):
                    raise ValueError("Não é possível realizar ajuste logarítimico com x <= 0")

                # aplicando o logarítmo natural aos valores de x
                ln_x = np.log(media)

                # obtendo os coeficientes e matriz de covariância
                coeficientes, covariancia = np.polyfit(ln_x, coordenadas_y, 1, cov=True)
                a, b = coeficientes

                # curva ajustada
                y_fit = a*np.log(media) + b

                # criando pontos adicionais para suavizar a curva
                x_suave = np.linspace(min(media), max(media), 500)
                y_suave = a*np.log(x_suave) + b

                # parâmetros utilizados para obter os limites das bandas de confiança e predição
                n = len(media)
                residuos = coordenadas_y - y_fit

                # coeficiente de determinação
                R2 = calcular_r2(coordenadas_y, y_fit)
                print(f"R² = {R2:.6f}")
                
                # dimensões e resolução da imagem
                plt.figure(figsize=(9,6), dpi=120)

                # plotando a curva ajustada
                plt.plot( x_suave, y_suave, label='Ajuste logarítmico', color = '#D55E00')

                # plotando os dados experimentais
                plt.scatter(media, coordenadas_y, label='Dados experimentais', zorder=3, color='#000000')

                # barras de erro associadas às medidas no eixo x com base no desvio padrão
                plt.errorbar(media, coordenadas_y, xerr=desvio_padrao, fmt='none', elinewidth=1.5, alpha=0.7, capsize=5, label='Desvio padrão', ecolor='#000000')
                
            elif informa_dados_analise_erros == "ordenadas":
                # mensagem de aviso caso algum dado considerado para o logarítmo seja negativo
                if any(x <= 0 for x in coordenadas_x):
                    raise ValueError("Não é possível realizar ajuste logarítimico com x <= 0")

                # aplicando o logarítmo natural aos valores de x
                ln_x = np.log(coordenadas_x)

                # obtendo os coeficientes e matriz de covariância
                coeficientes, covariancia = np.polyfit(ln_x, media, 1, cov=True)
                a,b = coeficientes

                # curva ajustada
                y_fit = a*np.log(coordenadas_x) + b

                # criando pontos adicionais para suavizar a curva
                x_suave = np.linspace(min(coordenadas_x), max(coordenadas_x), 500)
                y_suave = a*np.log(x_suave) + b

                # parâmetros utilizados para obter os limites das bandas de confiança e predição
                n = len(coordenadas_x)
                residuos = media - y_fit

                # coeficiente de determinação
                R2 = calcular_r2(media, y_fit)
                print(f"R² = {R2:.6f}")
                
                # dimensões e reoslução da imagem
                plt.figure(figsize=(9,6), dpi=120)

                # plotando a curva ajustada
                plt.plot( x_suave, y_suave, label='Ajuste logarítmico', color = '#D55E00')

                # plotando os dados experimentais
                plt.scatter(coordenadas_x, media, label='Dados experimentais', zorder=3, color='#000000')

                # barras de erro associadas às medidas no eixo y com base no desvio padrão
                plt.errorbar(coordenadas_x, media, yerr=desvio_padrao, fmt='none', elinewidth=1, alpha=0.7, capsize=5, label='Desvio padrão', ecolor='#000000')
                
            elif informa_dados_analise_erros == "ambas":
                # mensagem de aviso caso algum dado considerado para o logarítmo seja negativo
                if any(x <= 0 for x in media_x):
                    raise ValueError("Não é possível realizar ajuste logarítimico com x <= 0")

                # aplicando o logarítmo aos valores de x
                ln_x = np.log(media_x)

                # obtendo os coeficientes e matriz de covariância
                coeficientes, covariancia = np.polyfit(ln_x, media_y, 1, cov=True)
                a, b = coeficientes

                # curva ajustada
                y_fit = a*np.log(media_x) + b

                # criando pontos adicionais para suavizar a curva
                x_suave = np.linspace(min(media_x), max(media_x), 500)
                y_suave = a*np.log(x_suave) + b

                # parâmetros utilizados para obter os limites das bandas de confiança e predição
                n = len(media_x)
                residuos = media_y - y_fit

                # coeficiente de determinação
                R2 = calcular_r2(media_y, y_fit)
                print(f"R² = {R2:.6f}")
                
                # dimensões e resolução da imagem
                plt.figure(figsize=(9,6), dpi=120)

                # plotando a curva ajustada
                plt.plot( x_suave, y_suave, label='Ajuste logarítmico', color = '#D55E00')

                # plotando os dados experimentais
                plt.scatter(media_x, media_y, label='Dados experimentais', zorder=3, color='#000000')

                # barras de erro associadas às medidas em ambos os eixos com base no desvio padrão
                plt.errorbar(media_x, media_y, xerr=desvio_padrao_x, yerr=desvio_padrao_y, fmt='none', elinewidth=1.5, alpha=0.7, capsize=5, label='Desvio padrão', ecolor='#000000')

            if informa_dados_analise_erros == "nenhuma":
                # mensagem de aviso caso algum dado considerado para o logarítmo seja negativo
                if any(x <= 0 for x in coordenadas_x):
                    raise ValueError("Não é possível realizar ajuste logarítimico com x <= 0")

                # aplicando o logarítmo natural aos valores de x
                ln_x = np.log(coordenadas_x)

                # obtendo os coeficientes e matriz de covariância
                coeficientes, covariancia = np.polyfit(ln_x, coordenadas_y, 1, cov=True)
                a, b = coeficientes

                # curva ajustada
                y_fit = a*np.log(coordenadas_x) + b

                # criando pontos adicionais para suavizar a curva
                x_suave = np.linspace(min(coordenadas_x), max(coordenadas_x), 500)
                y_suave = a*np.log(x_suave) + b
                
                # coeficiente de determinação
                R2 = calcular_r2(coordenadas_y, y_fit)
                print(f"R² = {R2:.6f}")
                    
                # dimensões e resolução da imagem
                plt.figure(figsize=(9,6), dpi=120)

                # plotando a curva ajustada
                plt.plot( x_suave, y_suave, label='Ajuste logarítmico', color = '#D55E00')

                # plotando os dados experimentais
                plt.scatter(coordenadas_x, coordenadas_y, label='Dados experimentais', zorder=3, color='#000000')

            # obtendo as incertezas associadas aos parâmetros a e b com base na matriz de covariância
            erro_a = np.sqrt(covariancia[0,0])
            erro_b = np.sqrt(covariancia[1,1])
            
            print(f"Os coeficientes obtidos para a equação aln(x) + b = y que expressa essa função logarítmica são: a = {a:.3f} ± {erro_a:.3f}, b = {b:.3f} ± {erro_b:.3f}")
            
            print(f'Equação: y = ({a:.3f} ± {erro_a:.3f})ln(x) + ({b:.3f} ± {erro_b:.3f})')
            
            
            ajuste = "logarítmico"
            print(f'{avaliar_ajuste(R2,ajuste)}')

            if informa_dados_analise_erros in ["abscissas", "ordenadas", "ambas"]:
                x_medio = np.mean(ln_x)
    
                # dispersão das medidas para x
                Sxx = np.sum((ln_x - x_medio)**2)
                ln_x_suave = np.log(x_suave)
                
                graus_liberdade = n-2
                confianca = 0.95
    
                # dispersão dos resíduos para a curva ajustada
                s = np.sqrt(np.sum(residuos**2)/(graus_liberdade))
    
                # uso da distribuição t de Student para determinar o valor crítico da distribuição dos dados
                t_critico = t.ppf((1+confianca)/2, graus_liberdade)
    
                # erro associado à predição
                erro_pred = s*np.sqrt(1+1/n +(ln_x_suave - x_medio)**2/Sxx)
                pred_superior = y_suave + t_critico*erro_pred
                pred_inferior = y_suave - t_critico*erro_pred
    
                # erro associado à confiança
                erro_ic = s*np.sqrt(1/n + (ln_x_suave - x_medio)**2/Sxx)
                ic_superior = y_suave + t_critico*erro_ic
                ic_inferior = y_suave - t_critico*erro_ic

        """Adicionando a banda escolhida ao gráfico (para o caso em que há análise de erros)"""
        if informa_dados_analise_erros in ["ordenadas", "abscissas", "ambas"]:
            if banda == "predição":
                plt.fill_between(x_suave, pred_inferior, pred_superior, alpha=0.25, label='Banda de predição (95%)', color='#56B4E9')
                    
            if banda == "confiança":
                plt.fill_between(x_suave, ic_inferior, ic_superior, alpha=0.45, label='Banda de confiança (IC = 95%)', color='#0072B2')
                
            if banda == "ambas":
                plt.fill_between(x_suave, pred_inferior, pred_superior, alpha=0.25, label='Banda de predição (95%)', color='#56B4E9')
    
                plt.fill_between(x_suave, ic_inferior, ic_superior, alpha=0.45, label='Banda de confiança (IC = 95%)', color='#0072B2')
        
        # fontes e seus tamanhos para o gráfico
        plt.rcParams.update({
            "font.family": "serif",
            "font.size": 12,
            "axes.labelsize": 13,
            "axes.titlesize": 14,
            "legend.fontsize": 11
        })
        plt.xlabel(nome_eixo_abscissas)
        plt.ylabel(nome_eixo_ordenadas)
        plt.title(titulo_grafico)
        plt.legend()
        plt.grid(color = 'gray', linestyle = '--', linewidth = 0.5, alpha=0.35)
        plt.tight_layout()

        # condição de exportação da imagem
        if botao_salvar.value == "sim":
            nome = nome_arquivo_grafico.value

            # adicionando o formato de exportação caso o usuário não o faça
            if not nome.endswith('.png'):
                nome += '.png'
            
            plt.savefig(nome, dpi=600,bbox_inches='tight')
            print(f"Gráfico salvo como {nome}")
            
        else:
            print("O gráfico não foi salvo!")
            
        plt.show()

# aplicar a função e gerar o gráfico quando o botão for selecionado
botao_gerar.on_click(gerar_grafico)

# opções presentes na interface apresentada ao usuário
interface = widgets.VBox([
    tipo_ajuste,
    tipo_banda,
    botao_salvar,
    nome_arquivo_grafico,
    botao_gerar,
    saida
])

# exibindo a interface
display(interface)

---

## Conclusão

Portanto, com base no que foi exposto, é possível verificar que o projeto desenvolvido atende ao objetivo que se propôs, realizando a análise das incertezas de dados experimentais e gerando gráficos compatíveis com as medidas. A automatização dos cálculos e geração dos gráficos apresenta amplo potencial para minimizar erros associados à cálculos manuais, além de reduzir o tempo empregado para esse processo. Limitações do código estão associadas aos tipos de ajuste avaliados, que abragem apenas 4 categorias, o que pode ser melhorado em possíveis versões futuras.

No desenvolvimento do projeto, foi possível comprovar os aprendizados que se deram ao longo da disciplina de Práticas em Ciência de Dados, no entanto, dificuldades e desafios ainda foram observados ao implementar os códigos para as etapas de recebimento e tratamento dos dados e sua aplicação aos parâmetros estatísticos. Além disso, a necessidade de aprender a lógica matemática exigida pela implementação dos ajustes e das bibliotecas para as interfaces e gráficos demandaram tempo e muitas tentativas para o aprendizado.

Em síntese, o desenvolvimento da atividade foi satisfatório e útil para verificar e consolidar todos os conceitos vistos na disciplina de Práticas em Ciência de Dados, além de culminar em um projeto útil a ser utilizado em outras disciplinas e análises experimentais futuras.

---

## Referências

#### Referências que foram usadas como base para o desenvolvimento da lógica estatística e matemática

Hoffmann, R., & Univeridade de São Paulo. Escola Superior de Agricultura Luiz de Queiroz. (2017). Análise de regressão: uma introdução à econometria. Univeridade de São Paulo. Escola Superior de Agricultura Luiz de Queiroz.

De, F. L. P. (s. d.). Modelo de regressão linear Teoria e aplicações. Ufpr.br. Acesso em 18 de junho de 2026, de http://leg.ufpr.br/~lucambio/Linear/Regressao.pdf

Dalson Figueiredo Filho, Felipe Nunes, Enivaldo Rocha, Manoel Santos, Mariana Batista, José Alexandre Junior. (2011). O que fazer e o que não fazer com a regressão: pressupostos e aplicações do modelo linear de mínimos quadrados ordinários (MQO) [Data set]. Harvard Dataverse.

#### Referências usadas para nortear a aplicação das bibliotecas

Simple Widget Introduction — Jupyter Widgets 8.1.8 documentation. (s. d.). Readthedocs.Io. Acesso em 18 de junho de 2026, de https://ipywidgets.readthedocs.io/en/latest/examples/Widget%20Basics.html

Module: display — IPython 9.14.1 documentation. (s. d.). Readthedocs.Io. Acesso em 18 de junho de 2026, de https://ipython.readthedocs.io/en/stable/api/generated/IPython.display.html

W3schools.com. (s. d.). W3schools.com. Acesso em 18 de junho de 2026, de https://www.w3schools.com/python/numpy/numpy_intro.asp

W3schools.com. (s. d.-b). W3schools.com. Acesso em 18 de junho de 2026, de https://www.w3schools.com/python/scipy/index.php

W3schools.com. (s. d.-c). W3schools.com. Acesso em 18 de junho de 2026, de https://www.w3schools.com/python/matplotlib_intro.asp

Pyplot tutorial — Matplotlib 3.11.0 documentation. (s. d.). Matplotlib.org. Acesso em 18 de junho de 2026, de https://matplotlib.org/stable/tutorials/pyplot.html

---